# Which requirement forced which box

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 43 §43.1–43.5 · type: concept-lab

**The promise:** by the end you can take a *ranked requirement profile* and watch the right reference architecture — enterprise RAG assistant, autonomous workflow/ops agent, customer-facing copilot, or batch pipeline — fall out of it, name that architecture's **hardest problem** and **signature pattern**, and jump straight to the blueprint that implements it.

This notebook is fully offline and free: no API key, no services, nothing beyond the standard library. The whole chapter is a *reference* chapter — the four architectures already live as blueprints elsewhere in the repo — so this lab does not re-build them. It drives home the one lesson the chapter actually teaches: **the ranked requirements, not the technology, chose every box** (§43.5).

## 🧠 Why this matters

Most engineers collect architectures the way they collect framework logos — "we use Temporal," "we have a vector DB" — without being able to say which *requirement* forced each expensive box to exist. That is how a team ends up running a durable workflow engine for a chatbot, or a streaming index for a job that runs once a night.

Chapter 42 gave you the *method* for estimating and choosing; this chapter gives you the *answers* four common shapes already paid for. The trick to internalizing them is to read them backwards: not "here is the RAG architecture," but "**permissions** were ranked #1, and permissions are what made the ACL-filtered retrieval box exist." Flip the top requirement and a different box — a different architecture — must appear.

So this lab encodes the chapter's comparison table as a tiny data structure, gives you a router that maps a ranked-requirement profile to its architecture, and lets you *flip one requirement and watch the answer change*. The payload is the cross-links: each architecture points hard at the blueprint that realizes it, so this chapter is your index from "which shape?" to "study and lift the real one."

## Objectives & prereqs

**By the end you can:**
- Given a ranked requirement set, identify which of the four reference architectures fits, and name its *hardest problem* and *signature pattern*.
- Reproduce §43.5's four-row comparison table as a small, queryable data structure.
- Run a router that flips an architecture when you flip its top requirement — and predict the new box before the router answers.
- Reproduce the ADR-044 cost logic (≈ \$0.55/user/mo frontier-only, 78% small-model match, ~70% reduction) that *forces* tiered routing for the copilot.
- Apply the autonomy-dial rule that Ch 44 reuses for a shadow-mode launch.

**Prereqs:** Ch 42 read (the method — this chapter is *the answers* to it; run its estimator first). Helpful background: Ch 13 (RAG), Ch 31/33 (durable workflows), Ch 39–40 (gateway/caching), Ch 15 (structured outputs).

**Packages:** standard library only. No external dependencies, no API key, no network. Table-driven reasoning + tiny estimation; fully deterministic in CI.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import random
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default) keeps everything offline, free, and deterministic. This notebook
# is mock-only by design: it is pure table lookup + arithmetic, so there is no live
# path to enable. The switch is here so the pattern matches every other notebook in
# the repo — you flip it to 0 elsewhere once a real model call earns its cost.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed everything stochastic so the autonomy-dial simulation is identical every run.
SEED = 43
random.seed(SEED)

print(f"MOCK mode: {MOCK}  (offline, free, deterministic)")
print("No API key required. Nothing leaves this machine.")

## 🧠 Mental model: requirement first, box second

Here is the chapter compressed into one sentence: *if no requirement forced a box, the box is fashion* (§43.5 senior lens). The four architectures are not four technologies you pick from a menu; they are four *rankings of requirements*, and each ranking forces a different hardest problem, which forces a different signature pattern.

We start by reproducing §43.5's comparison table exactly — architecture → hardest problem, signature pattern, top risk — as a small data structure, plus, for each one, the *ranked requirement* that made it and the *blueprint* that implements it. Everything else in this notebook is a query over this one table.

In [ ]:
# The four reference architectures, compressed — §43.5's table, verbatim, as data.
# 'forcing_requirement' is the #1 ranked requirement that made each box exist (§43.1–43.4).

@dataclass(frozen=True)
class Architecture:
    key: str
    name: str
    forcing_requirement: str   # the #1 requirement that forced this shape
    hardest_problem: str       # §43.5 table, col 2
    signature_pattern: str     # §43.5 table, col 3
    top_risk: str              # §43.5 table, col 4
    blueprint: str             # relative path to the implementation to lift


ARCHITECTURES = {
    "enterprise_rag": Architecture(
        key="enterprise_rag",
        name="Enterprise RAG assistant",
        forcing_requirement="permissions",
        hardest_problem="Permissions",
        signature_pattern="ACL-filtered retrieval + citations",
        top_risk="Index/ACL drift; quiet quality decay",
        blueprint="../../../blueprints/rag-pipeline/",
    ),
    "workflow_ops": Architecture(
        key="workflow_ops",
        name="Autonomous workflow / ops agent",
        forcing_requirement="durability",
        hardest_problem="Durability + audit",
        signature_pattern="Durable workflow engine; autonomy dial",
        top_risk="Silent drift; runaway loops",
        blueprint="../../../blueprints/multi-agent-supervisor/",
    ),
    "copilot": Architecture(
        key="copilot",
        name="Customer-facing copilot at scale",
        forcing_requirement="latency_x_unit_cost",
        hardest_problem="Latency × unit cost",
        signature_pattern="Tiered routing gateway + caching",
        top_risk="Margin erosion; public abuse",
        blueprint="../../../blueprints/observability-stack/",
    ),
    "batch": Architecture(
        key="batch",
        name="Batch / agentic data pipeline",
        forcing_requirement="cost_per_item",
        hardest_problem="Cost per item",
        signature_pattern="Manifest + batch APIs + sampling QA",
        top_risk="Schema evolution; shipped errors",
        blueprint="../../../blueprints/eval-harness/",
    ),
}

# Print the table the way the book prints it.
hdr = f"{'Architecture':<34}{'Hardest problem':<22}{'Signature pattern':<40}{'Top risk'}"
print(hdr)
print("-" * len(hdr))
for a in ARCHITECTURES.values():
    print(f"{a.name:<34}{a.hardest_problem:<22}{a.signature_pattern:<40}{a.top_risk}")

## The four requirement→shape mappings

Each architecture is a *tight* requirement→shape story. Read each one as "the #1 requirement made this box; the box is the answer to that requirement, not a technology choice" (§43.1–43.4):

- **Enterprise RAG** — *permissions* ranked #1 → ACL-filtered retrieval (filter candidates against the caller's identity **before** any text reaches the model) + citations as the trust mechanism. Freshness is a 24h SLO, so ingestion stays a nightly batch + webhook nudges, not a streaming system.
- **Workflow / ops agent** — *durability + audit* ranked #1 → a durable workflow engine (Temporal / Step Functions) where every step is a retryable, persisted activity, plus an **autonomy dial** and an append-only audit log. No streaming UI — the frontend is a dashboard of runs.
- **Copilot at scale** — *latency × unit-cost* ranked #1 → a model **gateway** doing tiered routing (small model for the easy 80%, escalate on demand) + aggressive prompt-prefix and response caching. Cost engineering *is* product engineering here.
- **Batch pipeline** — *cost-per-item* ranked #1 → a per-item **manifest** in Postgres + **batch APIs** (~50% discount) + structured outputs validated against a schema + sampling QA against a golden set. State lives in rows, so the job is resumable and idempotent.

In [ ]:
# A requirement 'profile' is just a ranked list of requirements, most important first.
# The router reads the TOP requirement and returns the architecture it forces.
# This is a pure lookup over the table above — offline, deterministic, no model call.

# Map a forcing requirement -> the architecture key it produces.
_FORCES_BOX = {a.forcing_requirement: a.key for a in ARCHITECTURES.values()}

# Synonyms so a human-written profile resolves to a canonical requirement token.
_ALIASES = {
    "permissions": "permissions", "acl": "permissions", "access-control": "permissions",
    "durability": "durability", "auditability": "durability", "audit": "durability",
    "correctness": "durability",
    "latency": "latency_x_unit_cost", "unit-economics": "latency_x_unit_cost",
    "latency_x_unit_cost": "latency_x_unit_cost", "unit-cost": "latency_x_unit_cost",
    "cost-per-item": "cost_per_item", "cost_per_item": "cost_per_item",
    "throughput": "cost_per_item",
}


def route(profile: list[str]) -> Architecture:
    """Map a ranked requirement profile to the architecture its #1 requirement forces.

    The whole lesson of §43.5 in one function: the architecture is decided by the
    requirement at the *top* of the ranking, not by the technologies you like.
    """
    if not profile:
        raise ValueError("a requirement profile must rank at least one requirement")
    top_raw = profile[0].strip().lower()
    top = _ALIASES.get(top_raw)
    if top is None:
        raise ValueError(
            f"unknown requirement {profile[0]!r}; known: {sorted(set(_ALIASES))}"
        )
    return ARCHITECTURES[_FORCES_BOX[top]]


# The book's four canonical scenarios, as ranked profiles.
SCENARIOS = {
    "internal knowledge assistant": ["permissions", "trustworthiness", "freshness", "latency"],
    "headless invoice triage":      ["durability", "auditability", "throughput", "autonomy"],
    "in-product chat copilot":      ["latency", "unit-economics", "safety", "multi-tenancy"],
    "two-million-contract backfill":["cost-per-item", "throughput", "quality", "resumability"],
}

for scenario, profile in SCENARIOS.items():
    arch = route(profile)
    print(f"{scenario:<32} -> {arch.name}")
    print(f"{'':<32}    hardest: {arch.hardest_problem} | pattern: {arch.signature_pattern}")
    print(f"{'':<32}    lift it from: {arch.blueprint}")

## 🔮 Predict: flip the top requirement

The internal knowledge assistant routed to **Enterprise RAG** because *permissions* ranked #1. Now imagine the business changes: the same "answer from our documents" job is reframed as a **one-off backfill** — extract structured answers from two million archived documents into a warehouse, where **cost-per-item** is suddenly the #1 requirement and there is no live user identity in the loop at all.

**Predict, before you run the next cell:** which architecture does the profile become when you move `cost-per-item` to the top? Name its *hardest problem* and *signature pattern*. Write your guess down.

Then run the cell.

In [ ]:
original = SCENARIOS["internal knowledge assistant"]
flipped = ["cost-per-item", *[r for r in original if r != "latency"]]

before = route(original)
after = route(flipped)

print(f"profile (before): {original}")
print(f"  -> {before.name}  |  hardest: {before.hardest_problem}")
print(f"     signature pattern: {before.signature_pattern}\n")

print(f"profile (after) : {flipped}")
print(f"  -> {after.name}  |  hardest: {after.hardest_problem}")
print(f"     signature pattern: {after.signature_pattern}")
print(f"     lift it from: {after.blueprint}")

**What you just saw.** Changing *one* requirement — not one technology, one *requirement* — turned a RAG assistant into a batch pipeline. The vector store and the documents barely moved; what moved was the hardest problem (permissions → cost per item) and therefore the signature pattern (ACL-filtered retrieval → manifest + batch APIs + sampling QA). This is the chapter's whole claim made literal: **the ranked requirements chose the box.** When someone proposes an architecture, the question is always *which requirement forced each expensive box* — and if none did, the box is fashion.

## The autonomy dial (§43.2): a switch you *earn*, per action

The single most reusable pattern in the chapter is the **autonomy dial** from the workflow/ops architecture. You do not flip autonomy on; you *earn* it, per action type, with evidence. Launch with the agent **proposing** and humans **approving** everything; measure the agreement rate per action type; **promote an action to autonomous only when its agreement rate clears a threshold you trust** — and keep auditing samples after.

Below we simulate per-action agreement rates and apply that exact rule. This is the seed Ch 44 reuses to launch the capstone agent in shadow mode.

In [ ]:
# Simulate a few weeks of proposals: for each action type, how often did the agent's
# proposal match the human's decision? (MOCK: drawn from canned 'true' agreement rates
# plus sampling noise, seeded so CI is deterministic.)

TRUE_AGREEMENT = {                  # the unobserved real agreement rate per action
    "tag_invoice_category": 0.97,   # easy, well-bounded -> should promote
    "match_invoice_to_po":  0.93,   # mostly clean -> borderline at a strict bar
    "approve_payment":      0.71,   # money-touching, ambiguous -> stays human
    "flag_for_fraud":       0.62,   # high-stakes, noisy -> stays human
}
SAMPLES_PER_ACTION = 200            # how many proposal/decision pairs we reviewed
PROMOTE_THRESHOLD = 0.95           # the agreement bar to promote an action to autonomous


def observed_agreement(true_rate: float, n: int) -> float:
    """Sample n proposal/decision pairs; return the observed agreement fraction."""
    matches = sum(1 for _ in range(n) if random.random() < true_rate)
    return matches / n


def autonomy_decision(action: str, true_rate: float) -> dict:
    obs = observed_agreement(true_rate, SAMPLES_PER_ACTION)
    promote = obs >= PROMOTE_THRESHOLD
    return {
        "action": action,
        "observed_agreement": round(obs, 3),
        "mode": "AUTONOMOUS" if promote else "human-in-the-loop",
    }


print(f"promote-to-autonomous threshold: {PROMOTE_THRESHOLD:.0%}  (audit samples continue regardless)\n")
print(f"{'action':<24}{'agreement':>11}{'  decision'}")
print("-" * 48)
for action, true_rate in TRUE_AGREEMENT.items():
    d = autonomy_decision(action, true_rate)
    print(f"{d['action']:<24}{d['observed_agreement']:>11.1%}  {d['mode']}")

**What you just saw.** Autonomy became a *measured, reversible, per-action* decision rather than a debate. The two easy, well-bounded actions cleared the 95% bar and were promoted; the two high-stakes, money-touching actions did not, so they keep routing to the human escalation queue — enforced *structurally* (the autonomous tool set simply does not contain those verbs), not by a prompt asking the agent to be careful. Move `PROMOTE_THRESHOLD` and you move the line; that single knob is how you ship autonomy without incident.

## Cost-as-product (§43.3): reproducing the ADR-044 logic

The copilot's #1 requirement — *latency × unit-cost* — is not a vibe; it's arithmetic, and the arithmetic *forces* tiered routing. ADR-044 records the decision. Let's reproduce its numbers with a back-of-envelope estimator (the same shape as Ch 42's): frontier-only projects to **≈ \$0.55/user/month**, evals show the small model matches the frontier model on **78%** of real turns, and routing the easy majority to the small model yields **~70%** projected cost reduction — the difference between a feature that survives finance review and one that doesn't.

In [ ]:
# ADR-044 back-of-envelope. Illustrative, rounded prices — the POINT is the ratio, not
# the absolute dollars. Plug in your own per-1M-token prices to re-run for your stack.

TURNS_PER_USER_MONTH = 60        # rough: a couple of turns/day for an active user
TOKENS_PER_TURN = 1_500          # prompt + completion, after prefix caching
FRONTIER_USD_PER_1M = 6.00       # illustrative frontier blended price / 1M tokens
SMALL_USD_PER_1M = 0.60          # ~10x cheaper small model
SMALL_MATCH_RATE = 0.78          # offline evals: small model matches frontier on 78% of turns
PLAN_PRICE = 12.00               # the $12/month plan the cost must fit under


def cost_per_user_month(price_per_1m: float, fraction: float = 1.0) -> float:
    tokens = TURNS_PER_USER_MONTH * TOKENS_PER_TURN * fraction
    return tokens / 1_000_000 * price_per_1m


frontier_only = cost_per_user_month(FRONTIER_USD_PER_1M)

# Tiered routing: SMALL_MATCH_RATE of turns go to the small model, the rest to frontier.
tiered = (
    cost_per_user_month(SMALL_USD_PER_1M, SMALL_MATCH_RATE)
    + cost_per_user_month(FRONTIER_USD_PER_1M, 1 - SMALL_MATCH_RATE)
)
reduction = 1 - tiered / frontier_only

print(f"frontier-only:   ${frontier_only:5.2f} / user / month   ({frontier_only / PLAN_PRICE:.0%} of a ${PLAN_PRICE:.0f} plan's price)")
print(f"tiered routing:  ${tiered:5.2f} / user / month")
print(f"projected reduction: {reduction:.0%}  (small model matches frontier on {SMALL_MATCH_RATE:.0%} of turns)")
print()
verdict = "tiered routing is FORCED" if frontier_only > 0.4 else "frontier-only may be affordable"
print(f"verdict: {verdict} — this is the seed of ADR-044.")

**What you just saw.** The numbers, not taste, made the decision. Frontier-only lands around half a dollar per user per month — a small number until you multiply by a million users against a \$12 plan, at which point it dominates the margin. Because offline evals show a 10x-cheaper model is *good enough on 78% of turns*, tiered routing drops projected spend by roughly 70%. That is ADR-044's reasoning, and it is why **cost engineering is product engineering** in a consumer copilot: the route-by-difficulty gateway is what decides whether the feature survives its first finance review.

## ⚠️ Pitfall: the enterprise-RAG breach pattern

The breach in enterprise RAG is always the same story (§43.1): the index is built with a **privileged service account**, retrieval **ignores document ACLs**, and the assistant cheerfully summarizes the executive-compensation file to anyone who asks nicely. The fatal misconception is thinking you can fix this in the prompt — telling the model "only answer if the user is allowed." **The model cannot unsee what retrieval hands it**, and prompt instructions are not access control (Ch 41).

The fix is structural: **filter at retrieval, before any text reaches the model**, against the caller's identity. Below, the broken version filters after generation (too late — the secret already reached the model); the correct version filters the candidate set first.

In [ ]:
# A tiny ACL-aware retrieval mock. Each doc carries the groups allowed to see it —
# the 'load-bearing detail' is that ACLs ride INTO the index as metadata (§43.1).

INDEX = [
    {"id": "handbook",   "allowed": {"all-staff"},          "text": "PTO accrues at 1.5 days/month."},
    {"id": "comp-execs", "allowed": {"exec", "hr"},          "text": "CEO total comp: $4.2M."},
    {"id": "roadmap",    "allowed": {"product", "exec"},     "text": "Q3 launches: project Atlas."},
]


def retrieve_broken(query_groups: set) -> list[str]:
    """BROKEN: retrieve everything, hand it ALL to the model, 'filter' after. Too late."""
    candidates = [d["text"] for d in INDEX]            # ignores ACLs at retrieval
    answer = " ".join(candidates)                      # the model has now SEEN every secret
    return [answer]


def retrieve_acl_filtered(query_groups: set) -> list[str]:
    """CORRECT: filter candidates by the caller's groups BEFORE any text reaches the model."""
    return [d["text"] for d in INDEX if d["allowed"] & query_groups]


caller_groups = {"all-staff"}   # an ordinary employee, NOT exec/hr/product

print("BROKEN (filter-after-generation) — the model saw:")
print("  ", retrieve_broken(caller_groups)[0])
print("  ^ the comp file leaked: it reached the model, which cannot unsee it.\n")

print("CORRECT (ACL-filtered retrieval) — the model only ever saw:")
for chunk in retrieve_acl_filtered(caller_groups):
    print("  ", chunk)
print("  ^ secrets were never retrieved, so they could never be summarized.")

**What you just saw.** Same documents, same model, two outcomes — decided entirely by *where* the permission check sits. In the broken path the secret reaches the model and no downstream instruction can claw it back; in the correct path the secret is never retrieved, so it can never leak. This is why §43.1 insists permissions are enforced in the retrieval query, why ACL changes must sync on the same SLO as content, and why the eval suite needs **permission probes** — queries that should return *nothing*.

## 🎯 Senior lens

Read the four architectures as **one lesson**: the ranked requirements, not the technology, chose every shape (§43.5). Permissions made the RAG assistant; durability and audit made the ops agent; latency and unit economics made the copilot; cost-per-item made the batch pipeline. When someone proposes an architecture, ask which requirement forced each expensive box. If no requirement did, the box is **fashion** — and fashion is how teams end up running a workflow engine for a chatbot or a streaming index for nightly data.

The second half of the senior move is the paper trail. A reference architecture borrowed *without recorded reasoning* is cargo cult; the same architecture with **ADRs for the deltas** — why Temporal over Celery, why ACL-filtered retrieval over per-team indexes, why the router defaults to the small model — is judgment you can defend, revisit, and hand to the next engineer. The deltas *are* your ADRs ([`../../../templates/adr-template/`](../../../templates/adr-template/)). That habit, not the boxes, is what this chapter is actually teaching.

## Recap

- The four reference architectures are four *rankings of requirements*, not four technologies: the #1 requirement forces the hardest problem, which forces the signature pattern.
- A one-line **router** over §43.5's table maps a ranked profile to its architecture — and flipping the top requirement flips the architecture (RAG → batch).
- The **autonomy dial** makes autonomy a measured, reversible, per-action decision: promote an action only when its agreement rate clears a trusted threshold, and keep auditing.
- For the copilot, **cost is product**: the ADR-044 arithmetic (≈ \$0.55/user/mo frontier-only, 78% small-model match) *forces* tiered routing for ~70% savings.
- The enterprise-RAG breach is always *filter-too-late*: enforce permissions **at retrieval**, because the model cannot unsee what retrieval hands it.
- A borrowed architecture without ADRs for its deltas is cargo cult; the deltas are your ADRs.

## Exercises

Each exercise *changes one requirement* and asks you to predict which box must change before you run it.

1. **Flip durability onto the copilot.** Take the `in-product chat copilot` profile and move a hard `durability`/`auditability` requirement to #1 (say it now drives money-moving back-office actions). Predict the new architecture and its hardest problem, then run `route(...)` to check. Which signature pattern did you just inherit — and which blueprint do you now lift from?
2. **Move the autonomy bar.** Change `PROMOTE_THRESHOLD` to `0.90` and re-run the autonomy-dial cell. Predict which action(s) flip to `AUTONOMOUS`. Then argue: for `approve_payment`, is *any* agreement threshold enough to remove the human, or is this a requirement that structurally forbids autonomy regardless of the number?
3. **Break the cost case.** In the ADR-044 cell, find the `SMALL_MATCH_RATE` at which tiered routing's projected reduction drops below ~30%. Below that match rate, is tiered routing still worth its added complexity (a router to monitor, evaluate, tune)? Write the one-sentence ADR consequence you'd record.
4. **Add a permission probe.** Extend the ACL pitfall cell with a query a `{'all-staff'}` caller issues that *should* return nothing from `retrieve_acl_filtered` (e.g. "what is the CEO's compensation?"). Assert the result is empty — that assertion is exactly the kind of permission-probe eval §43.1 says to add to the suite.

In [ ]:
# Exercise scratch space — your code here.

In [ ]:
# Exercise scratch space — your code here.

## Next

- **This chapter is an index, not a build.** Each architecture points hard at the blueprint that realizes it — go study and lift the real one:
  - Enterprise RAG → [`../../../blueprints/rag-pipeline/`](../../../blueprints/rag-pipeline/) (ACL-aware retrieval, citations), with faithfulness/permission-probe evals from [`../../../blueprints/eval-harness/`](../../../blueprints/eval-harness/).
  - Workflow / ops agent → [`../../../blueprints/multi-agent-supervisor/`](../../../blueprints/multi-agent-supervisor/) + the durable-run/idempotency seams of [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/).
  - Copilot at scale → the gateway / tiered-routing cost layer in [`../../../blueprints/observability-stack/`](../../../blueprints/observability-stack/) (Ch 39–40), with guardrails from Ch 41.
  - Batch pipeline → the structured-output + sampling-QA patterns (Ch 15) and the golden-set sampling in [`../../../blueprints/eval-harness/`](../../../blueprints/eval-harness/).
- **Template:** the chapter's ADR-044 is a worked instance of [`../../../templates/adr-template/`](../../../templates/adr-template/) — *the deltas are your ADRs* is the template's reason to exist.
- **Capstone & next chapter:** these four architectures are the case studies that `capstone/` (an enterprise-RAG + workflow hybrid) instantiates; **Ch 44** then tours that instantiation end-to-end — and reuses this notebook's autonomy-dial rule to launch the capstone agent in shadow mode.
- **Book:** keep §43.5's comparison table within reach — *requirement first, box second* is the sentence to carry into every architecture review.